In [ ]:
# install elan + Lean 4 toolchain
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | bash -s -- -y --default-toolchain leanprover/lean4:v4.11.0
import os
os.environ['PATH'] = '/root/.elan/bin:' + os.environ['PATH']
!lean --version

In [ ]:
import os, json, subprocess
from pathlib import Path

WORK = Path('/kaggle/working/verify')
WORK.mkdir(exist_ok=True)

# locate harvested.jsonl from harvest-v2 kernel_sources mount
HARVEST = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'harvested.jsonl' in files:
        HARVEST = os.path.join(root, 'harvested.jsonl')
        break
assert HARVEST, 'harvested.jsonl not found'
print('HARVEST:', HARVEST)

# load first row with proofs
with open(HARVEST) as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'rows: {len(rows)}')
sample = rows[0]
print(f'sample id={sample["id"]} eq1={sample["eq1"]} eq2={sample["eq2"]} label={sample["label"]}')
print(f'proofs: {len(sample["proofs"])}')
for i, p in enumerate(sample['proofs']):
    print(f'  [{i}]: {p[:150]}')

In [ ]:
# build minimal Lean file with Magma class + theorem + proof, check via lean cli
import os, subprocess, time

PREAMBLE = '''class Magma (G : Type) where
  op : G \u2192 G \u2192 G
infixl:70 " \u25c7 " => Magma.op
'''

def to_diamond(s):
    return s.replace('*', '\u25c7')

def build_lean(eq1, eq2, proof_body):
    eq1d = to_diamond(eq1)
    eq2d = to_diamond(eq2)
    return (PREAMBLE +
        'theorem sair_implication\n'
        '    (G : Type) [inst : Magma G]\n'
        f'    (h : \u2200 x y z w u : G, {eq1d})\n'
        f'    : \u2200 x y z w u : G, {eq2d} := ' + proof_body + '\n')

def check_proof(lean_src, timeout=60):
    p = WORK / 'test.lean'
    p.write_text(lean_src)
    t0 = time.time()
    try:
        r = subprocess.run(['lean', str(p)], capture_output=True, text=True, timeout=timeout)
        return {'ok': r.returncode == 0, 'rc': r.returncode,
                'stdout': r.stdout[:500], 'stderr': r.stderr[:500],
                'elapsed': time.time() - t0}
    except subprocess.TimeoutExpired:
        return {'ok': False, 'rc': -1, 'stdout': '', 'stderr': 'TIMEOUT', 'elapsed': time.time()-t0}

for i, proof in enumerate(sample['proofs']):
    src = build_lean(sample['eq1'], sample['eq2'], proof)
    print(f'--- proof[{i}] ---')
    print('LEAN SRC:')
    print(src)
    res = check_proof(src)
    print(f'RESULT: ok={res["ok"]} rc={res["rc"]} elapsed={res["elapsed"]:.1f}s')
    if not res['ok']:
        print('stdout:', res['stdout'])
        print('stderr:', res['stderr'])
    print()